In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib as m
from tqdm import tqdm
from scipy import linalg
import scipy as sp
import os
import pickle
import glob
import h5py
import random

from scipy.constants import *

phi0 = physical_constants["mag. flux quantum"][0]
eps0 = epsilon_0

from scipy.special import eval_hermite
import scfitpy as Qfit

In [ ]:
F = glob.glob("./cwSpec_currentSweep_202407011423.hdf5")
with h5py.File(F[0], "r") as f:
    data = f["Data/Data"]
    current = data[0][1][300:1100] * 10**3
    freq = data[:, 0, 0][70:330] * 10 ** (-9)
    mag0 = data[:, 2, :].T[300:1100]
mag1 = []
for i in range(len(mag0)):
    mag1.append(mag0[i][70:330])
mag1 = np.array(mag1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plt.rcParams["font.size"] = 25
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"

X, Y = np.meshgrid(freq, current)
mappable = ax.pcolor(Y, X, Qfit.normalization(mag1), cmap="summer")
cbar_num_format = "%.2f"
cbar = plt.colorbar(mappable, ax=ax, format=cbar_num_format)

# ax.set_xlim([-12,12])
# ax.set_ylim([5,5.4])
# plt.grid()
plt.xlabel(r"Current [mA]", fontsize=25)
plt.ylabel(r"$\omega_p\ $[GHz]", fontsize=25)
# plt.savefig("IF100Hz.png",bbox_inches="tight", pad_inches=0.5,dpi=700)
plt.show()

In [ ]:
result = Qfit.photoProcess(mag1, 4, black_ridges=True)

In [ ]:
cont_list = Qfit.ContFind(result, 0.016, 40)

In [ ]:
c_list = list(m.colors.CSS4_COLORS.values())
random.shuffle(c_list)
xyp = [
    [np.mean(cont_list[i][:, 1]), np.mean(cont_list[i][:, 0])]
    for i in range(len(cont_list))
]
for i in range(len(cont_list)):
    for j in range(len(cont_list)):
        if i > j and abs(np.linalg.norm(xyp[i]) - np.linalg.norm(xyp[j])) < 10:
            xyp[j][1] = xyp[j][1] * 1.3
plt.rcParams["font.size"] = 25
fig, ax = plt.subplots(figsize=(8, 6))

for i in range(len(cont_list)):
    x = cont_list[i][:, 1]
    y = cont_list[i][:, 0]
    ax.plot(x, y, linewidth=2, color=c_list[i], label=i)
    ax.annotate(
        i,
        xy=tuple(xyp[i]),
        fontsize=10,
        color="k",
        bbox={"facecolor": c_list[i], "edgecolor": "w", "alpha": 0.3},
    )
# ax.legend(bbox_to_anchor=(1,1))
ax.set_xlabel("row index")
ax.set_ylabel("column index")
# plt.savefig("cont_list.png",bbox_inches="tight", pad_inches=0.5,dpi=500)
# plt.show()

In [ ]:
# band structure arrangement determination by hand
contour0 = np.concatenate([cont_list[2], cont_list[3]])
contour1 = np.concatenate([cont_list[7], cont_list[6]])
contour2 = np.concatenate([cont_list[4], cont_list[5]])

cont_band = [contour0, contour1, contour2]

In [ ]:
cur_p, band_min = Qfit.peakTrace(
    result, freq, current, cont_band, 0, 5, 4, black_ridges=False
)

In [ ]:
# with open('2Q_tracePT_band.pickle', mode='wb') as fo:
#     pickle.dump(band_min, fo)
# with open('2Q_tracePT_cur.pickle', mode='wb') as fo:
#     pickle.dump(cur_p, fo)

In [ ]:
f = open("2Q_tracePT_band.pickle", "rb")
band_min = pickle.load(f)
f = open("2Q_tracePT_cur.pickle", "rb")
cur_p = pickle.load(f)

#### from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(8, 6))
plt.rcParams["font.size"] = 25
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
X, Y = np.meshgrid(freq, current)
mappable = ax.pcolor(Y, X, Qfit.normalization(mag1), cmap="summer")
cbar_num_format = "%.2f"
cbar = plt.colorbar(mappable, ax=ax, format=cbar_num_format)

for i in range(len(band_min)):
    ax.plot(cur_p[i], band_min[i], "*", ms=5, label=str(i))

# ax.set_ylim([5,6])
ax.set_ylabel(r"$\omega_p$ [GHz]", fontsize=24)
ax.set_xlabel(r"Current [mA]", fontsize=24)
fig.tight_layout()
plt.savefig("TracedFig.png", bbox_inches="tight", pad_inches=0.5)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm

# 3Dプロットの準備
# fig = plt.figure(figsize=(10, 7))
fig, ax = plt.subplots(figsize=(2.5, 4))
clist_p = []

i = 0

n_values = [115]  # nの値
for n in n_values:
    i = i + 1
    c = cur_p[1][n]
    f = band_min[1][n]
    p = np.argmin(abs(c - current))
    v = Qfit.normalization(mag1)[p][np.argmin(abs(freq - f))]
    n_array = np.full_like(freq, c)

    plt.plot(
        freq,
        Qfit.normalization(mag1)[p] + 4 * 0.6,
        "-",
        lw=0.5,
        color=cm.hsv(i / 10),
        label=str(c),
    )
    plt.plot(f, v + 4 * 0.6, "*", color="black")
    clist_p.append(c)

n_values = [3, 8, 25]  # nの値

for n in n_values:
    i = i + 1
    c = cur_p[2][n]
    n2 = np.argmin(abs(c - cur_p[1]))
    f = band_min[2][n]
    p = np.argmin(abs(c - current))
    v = Qfit.normalization(mag1)[p][np.argmin(abs(freq - f))]
    n_array = np.full_like(freq, c)

    plt.plot(
        freq,
        Qfit.normalization(mag1)[p] + (i - 1) * 0.6,
        "-",
        lw=0.5,
        color=cm.hsv(i / 10),
        label=str(c),
    )
    plt.plot(f, v + (i - 1) * 0.6, "*", color="black")

    c2 = cur_p[1][n2]
    f2 = band_min[1][n2]
    p2 = np.argmin(abs(c2 - current))
    v2 = Qfit.normalization(mag1)[p2][np.argmin(abs(freq - f2))]
    n_array2 = np.full_like(freq, c2)

    # plt.plot(freq, Qfit.standardization(mag1)[p2], '-',lw=0.5,color=cm.hsv(i/10))
    plt.plot(f2, v2 + (i - 1) * 0.6, "*", color="black")
    clist_p.append(c)

# ラベルと凡例を設定
ax.set_ylabel("Amplitude", labelpad=15)
ax.set_xlabel(r"$\omega_p$ [GHz]", labelpad=15)
# ax.legend()

# ラベル位置をさらに改善
ax.xaxis.label.set_size(20)
ax.yaxis.label.set_size(20)
ax.tick_params(axis="both", which="major", labelsize=20)  # 軸目盛のサイズ調整
ax.tick_params(labelbottom=True, labelleft=False, labelright=False, labeltop=False)
ax.tick_params(bottom=True, left=False, right=False, top=False)

# fig.tight_layout(pad=0.1)  # 全体の余白を調整
plt.subplots_adjust(left=0, right=2, top=2, bottom=1)  # さらに微調整
# プロットを表示
plt.savefig("PT2D.png", bbox_inches="tight", pad_inches=0.5)
plt.show()
clist_p

In [ ]:
#### from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(8, 6))
plt.rcParams["font.size"] = 25
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
X, Y = np.meshgrid(freq, current)
mappable = ax.pcolor(Y, X, Qfit.normalization(mag1), cmap="summer")
cbar_num_format = "%.2f"
cbar = plt.colorbar(mappable, ax=ax, format=cbar_num_format)

for i in range(len(band_min)):
    ax.plot(cur_p[i], band_min[i], "*", ms=5, label=str(i))

for c in clist_p:
    plt.vlines(c, max(freq), 5.45, color="black", lw=1)

# ax.set_ylim([5,6])
ax.set_ylabel(r"$\omega_p$ [GHz]", fontsize=24)
ax.set_xlabel(r"Current [mA]", fontsize=24)
fig.tight_layout()
plt.savefig("TracedFig.png", bbox_inches="tight", pad_inches=0.5)
plt.show()